<a href="https://colab.research.google.com/github/mannduuu07-png/korea-fire-frequency-severity-analysis/blob/main/notebooks/01_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 - Data Cleaning

This notebook loads 10 years (2015-2024) of Korean national fire
statistics and prepares a clean, unified dataset for analysis.

**Data source:** 소방청 (National Fire Agency) 연간화재통계,
published annually via [data.go.kr](https://www.data.go.kr)

**Note:** Raw CSV files are not included in this repo due to size.
See `data/raw/README.md` for download instructions.

In [1]:
import pandas as pd
from scipy.stats import spearmanr
import glob

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

DATA_DIR = '/content/drive/MyDrive/fire_data'  # adjust to your path
files_list = glob.glob(f'{DATA_DIR}/*.csv')
print(f"Found {len(files_list)} files")
assert len(files_list) == 10, "Expected 10 yearly files (2015-2024)"

Mounted at /content/drive
Found 10 files


## Data Quality Issue #1: Inconsistent column names across years

The 10 yearly files were published by different people at different
times, and column names are not consistent:
- Date column: `일시` (2022-24) vs `화재발생년월일` (2015-21)
- Region column: `시_군_구` / `시·군·구` / `시군구` (three different spellings)
- Ignition source: `발화열원` (2015-18, single column) vs
  `발화열원대분류` + `발화열원소분류` (2019-24, split into two)

Column names must be unified **before** concatenating, or pandas
silently creates duplicate columns full of NaN.

In [2]:
RENAME_MAP = {
    '일시': '화재발생년월일',
    '시·군·구': '시군구',
    '시_군_구': '시군구',
    '발화열원': '발화열원대분류',
}

df_list = []
for f in files_list:
    temp = pd.read_csv(f, encoding='cp949')
    temp = temp.rename(columns=RENAME_MAP)
    if '연번' in temp.columns:
        temp = temp.drop(columns=['연번'])

    # Parse dates PER FILE, before concat -- pandas infers a single
    # date format from the first few rows; if formats differ across
    # files, parsing the concatenated column at once silently fails
    # for the files that don't match.
    temp['화재발생년월일'] = pd.to_datetime(temp['화재발생년월일'], errors='coerce')
    fail = temp['화재발생년월일'].isna().sum()
    if fail > 0:
        print(f"  Warning: {f.split('/')[-1]} has {fail} unparsed dates")

    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)
print(f"\nTotal rows: {len(df)}")
print(f"Unparsed dates: {df['화재발생년월일'].isna().sum()}")
print(f"Year range: {int(df['화재발생년월일'].dt.year.min())}-{int(df['화재발생년월일'].dt.year.max())}")


Total rows: 405977
Unparsed dates: 0
Year range: 2015-2024


## Data Quality Issue #2: Administrative boundary changes over 10 years

Korean administrative divisions changed during this period:
- Incheon's 남구 was renamed 미추홀구 on 2018-07-01
- Gunwi-gun's province affiliation may have changed (경북 → 대구, 2023-07-01)

Additionally, many district names (남구, 중구, 동구, etc.) are shared
by multiple cities nationwide (e.g. 중구 exists in 6 different cities).
This becomes critical later when comparing regions by name alone --
see `04_robustness_checks.ipynb` for a bug this caused.

In [7]:
# Strip whitespace inconsistencies (e.g. '청주시 상당구' vs '청주시상당구')
df['시군구'] = df['시군구'].str.replace(' ', '', regex=False).str.strip()

# Incheon 남구 -> 미추홀구 rename. Sido condition is required --
# without it, this would incorrectly rename 남구 in Busan/Daegu/
# Gwangju/Ulsan as well.
mask = (df['시도'] == '인천광역시') & (df['시군구'] == '남구')
df.loc[mask, '시군구'] = '미추홀구'

# 강원도 -> 강원특별자치도 (2023.6.11 개명, 경계 변경 없음 - 단순 rename)
df.loc[df['시도'] == '강원도', '시도'] = '강원특별자치도'

# 전라북도 -> 전북특별자치도 (2024.1.18 개명, 경계 변경 없음)
df.loc[df['시도'] == '전라북도', '시도'] = '전북특별자치도'

# 군위군: 경북 -> 대구 편입 (2023.7.1) -- 이전에 확인만 하고 미적용이었음, 이번엔 실제 적용
df.loc[df['시군구'] == '군위군', '시도'] = '대구광역시'

# 부천시: 2016.7.4 구 폐지 -> 2024.1.1 재설치, 중간 7년은 구분 없이 '부천시'로 존재했음
# 10년 일관 비교를 위해 전체 기간을 부천시 단일 단위로 통합
df.loc[(df['시도']=='경기도') & (df['시군구'].isin(['부천시원미구','부천시소사구','부천시오정구'])), '시군구'] = '부천시'

# 최종 확인 -- 이제 시도 목록에 구 명칭이 남아있으면 안 됨
print(df['시도'].unique())
print(f"\n원주시/춘천시 통합 확인:")
print(df[df['시군구'].isin(['원주시','춘천시'])].groupby(['시도','시군구']).size())

# Check Gunwi-gun's province affiliation
print("[Gunwi-gun by province]")
print(df[df['시군구'] == '군위군'].groupby('시도').size())
# If split across two provinces, uncomment to unify:
# df.loc[df['시군구']=='군위군', '시도'] = '대구광역시'

print(f"\nDuplicate rows: {df.duplicated().sum()}")

['경상북도' '서울특별시' '경기도' '충청남도' '부산광역시' '강원특별자치도' '세종특별자치시' '울산광역시' '경상남도'
 '전라남도' '인천광역시' '전북특별자치도' '대구광역시' '제주특별자치도' '충청북도' '광주광역시' '대전광역시']

원주시/춘천시 통합 확인:
시도       시군구
강원특별자치도  원주시    3218
         춘천시    2627
dtype: int64
[Gunwi-gun by province]
시도
대구광역시    473
dtype: int64

Duplicate rows: 2


In [10]:
df['연도'] = df['화재발생년월일'].dt.year
df['월'] = df['화재발생년월일'].dt.month

# Save cleaned data for use in subsequent notebooks
df.to_parquet('/content/drive/MyDrive/fire_data/processed/cleaned_fire_data.parquet', index=False)
print("Saved cleaned dataset.")

Saved cleaned dataset.
